# Why tiny local gains have not improved Kaggle

This notebook records a diagnostic review after two unsuccessful directions: global robustness-focused alternatives and longer boosting runs. It makes no changes to trained models, existing CSVs or the provided data. All leaderboard scores below were reported by the user; no hidden labels or Kaggle API calls were used.

The assignment permits disclosed pretrained weights, but authorship labels must come only from the supplied training set. External LLM APIs must not label the test passages. The current best remains `submission_final_model_improvements.csv` (reported Kaggle AUROC 0.98629).


In [1]:
import json, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import nbformat
from IPython.display import display

ROOT = Path.cwd().resolve()
OUT = ROOT / 'artifacts/validation_gap_diagnosis'
OUT.mkdir(parents=True,exist_ok=True)
records = [
    {'file':'submission_final_model_improvements.csv','kaggle_reported':.98629,'local_holdout':.999449,'change':'Original nine-specialist recipe'},
    {'file':'candidate_01.csv','kaggle_reported':.98379,'local_holdout':.999195,'change':'New representation/weights, two seeds'},
    {'file':'candidate_03.csv','kaggle_reported':.98326,'local_holdout':.999213,'change':'New representation/weights, two seeds'},
    {'file':'candidate_04.csv','kaggle_reported':.98363,'local_holdout':.999204,'change':'New representation/weights, two seeds'},
    {'file':'longer_01_lgb8_cat8_xgb8.csv','kaggle_reported':.98588,'local_holdout':.999486,'change':'Same winning recipe, longer training'},
]
scorecard = pd.DataFrame(records)
scorecard['reported_delta_from_best'] = scorecard.kaggle_reported-.98629
scorecard.to_csv(OUT/'reported_scores.csv',index=False)
display(scorecard)
assert scorecard.loc[scorecard.kaggle_reported.idxmax(),'file']=='submission_final_model_improvements.csv'


,file,kaggle_reported,local_holdout,change,reported_delta_from_best
0,submission_final_model_improvements.csv,0.98629,0.999449,Original nine-specialist recipe,0.00000
1,candidate_01.csv,0.98379,0.999195,"New representation/weights, two seeds",-0.00250
2,candidate_03.csv,0.98326,0.999213,"New representation/weights, two seeds",-0.00303
3,candidate_04.csv,0.98363,0.999204,"New representation/weights, two seeds",-0.00266
4,longer_01_lgb8_cat8_xgb8.csv,0.98588,0.999486,"Same winning recipe, longer training",-0.00041


## Observable genre support

Reuse the stored exploratory table, without refitting on test text. These categories are heuristic descriptions, not true domain labels. A short-line poetry rule can misclassify other kinds of lists or formatting. The support gap is a warning about validation coverage, not proof of why any one Kaggle score fell.


In [2]:
feature_nb = nbformat.read((ROOT/'feature_analysis.ipynb' if (ROOT/'feature_analysis.ipynb').exists() else ROOT/'plum/feature_analysis.ipynb'),as_version=4)
source = feature_nb.cells[19].source
assert 'domain_table' in source
# The pre-existing executed table is retained as evidence; no model is fitted here.
for output in feature_nb.cells[19].get('outputs',[]):
    plain = output.get('data',{}).get('text/plain','')
    if isinstance(plain,str) and 'Train All %' in plain and 'Test %' in plain:
        print(plain)
subgroups = pd.read_csv((ROOT/'artifacts/final_model_improvements/holdout_subgroups.csv' if (ROOT/'artifacts/final_model_improvements/holdout_subgroups.csv').exists() else ROOT/'plum/artifacts/final_model_improvements/holdout_subgroups.csv'))
genres = subgroups[subgroups.grouping=='heuristic_domain'].copy()
genres['auc_estimable'] = (genres.human>0)&(genres.AI>0)
genres.to_csv(OUT/'holdout_genre_support.csv',index=False)
display(genres[['group','rows','human','AI','auc','auc_estimable']])
poetry = genres[genres['group'].str.startswith('3.')].iloc[0]
assert poetry.rows==3 and poetry.human==0 and poetry.AI==3 and not poetry.auc_estimable


                              Train Human (0) %  Train AI (1) %  Train All %  \
text                                                                           
1. Recipe / Cooking                       20.72           20.92        20.82   
2. Academic / ArXiv Abstract              18.36           21.08        19.72   
3. Poetry / Verse / Lyrics                 0.02            0.22         0.12   
4. Book / Novel Summary                    3.12            4.88         4.00   
5. Film / TV / Review                      8.32            5.52         6.92   
6. Encyclopedia / Wiki Bio                12.16            9.54        10.85   
7. First-Person / Social                  17.34           17.16        17.25   
8. General / News / Other                 19.96           20.68        20.32   

                              Unlabeled %  Test %  Test vs Train Shift (pp)  
text                                                                         
1. Recipe / Cooking                 20.80  

,group,rows,human,AI,auc,auc_estimable
3,1. Recipe / Cooking,420,207,213,0.999909,True
4,2. Academic / ArXiv Abstract,370,183,187,0.999737,True
5,3. Poetry / Verse / Lyrics,3,0,3,NaN,False
6,4. Book / Novel Summary,77,31,46,0.990884,True
7,5. Film / TV / Review,137,79,58,1.000000,True
8,6. Encyclopedia / Wiki Bio,225,122,103,1.000000,True
9,7. First-Person / Social,340,173,167,0.999446,True
10,8. General / News / Other,428,205,223,0.999497,True


## Existing negative evidence

The original semi-supervised experiment selected zero student contribution. Repeating the same pseudo-labeling setup has little support. More boosting iterations improved local AUROC but the reported leaderboard score fell; the current validation ranking is therefore insufficient for choosing tiny parameter changes. Full formatting robustness also changed the model substantially and lowered the reported score. Avoid drawing causal conclusions from that multi-change comparison.


In [3]:
ssl = json.loads((ROOT/'artifacts/final_model_improvements/ssl_selected.json').read_text())
print('Previous semi-supervised selection:',ssl)
assert not ssl['enabled']
import torch, importlib.util
compute = {'python':sys.version.split()[0],'torch':torch.__version__,'torch_cuda_available':torch.cuda.is_available(),
           'transformers_installed':importlib.util.find_spec('transformers') is not None}
result = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
compute['nvidia_hardware'] = result.stdout.strip() if result.returncode==0 else None
(OUT/'compute.json').write_text(json.dumps(compute,indent=2),encoding='utf-8')
print(json.dumps(compute,indent=2))


Previous semi-supervised selection: {'enabled': False, 'setting': {'weight': 0.1, 'threshold': 0.95, 'agreement': False}, 'contribution': 0.0, 'reason': 'No confirmed gain beyond tolerance.', 'cv_gain': 0.0, 'passes_ood': True}


{
  "python": "3.14.6",
  "torch": "2.13.0+cpu",
  "torch_cuda_available": false,
  "transformers_installed": true,
  "nvidia_hardware": "NVIDIA GeForce RTX 5060 Laptop GPU, 8151 MiB"
}


## Recommended next experiments

1. **A genuinely different representation:** train a compact generic pretrained transformer on the supplied authorship labels, then evaluate a conservative blend with the original winning recipe. A small DeBERTa model is one candidate. First validate a small local pilot and compare complementarity, not just the new model's standalone AUROC. Preserve the winning CSV. The machine has an NVIDIA RTX 5060 Laptop GPU with about 8 GB VRAM, but the installed PyTorch build is CPU-only; GPU training is not configured. Use an isolated environment under this submission's artifacts rather than altering the existing environment. Pretrained weights and AI assistance must be disclosed.
2. **Low-risk control:** average 3–5 independently seeded fits of the original nine-specialist recipe at its original iteration counts. The earlier multi-seed run also changed the representation and weights, so it did not isolate seed averaging of the winning recipe. Keep all other settings fixed; evaluate original and shifted-domain slices before considering submission. Averaging is a variance-reduction hypothesis, not an assured score gain.
3. **Improve the selection signal:** emphasize development-only held-out genres and clearly defined perturbations, report sample counts, and consider cautious test-distribution-weighted diagnostics. Reweighting cannot create missing human-poetry labels, and importance weighting assumes sufficiently stable class-conditional relationships and support overlap. Do not claim that a weighted score validates unseen genres when those assumptions fail.

Generic pretrained encoders are permitted by the assignment, but an externally trained AI-detector checkpoint would require separate policy scrutiny and is not proposed here. No external AI labels, pretrained downloads, package installation, GPU setup, new training or new submission generation occurred in this diagnostic notebook.

References: [DeBERTa-v3-small model card](https://huggingface.co/microsoft/deberta-v3-small), [Transformers text classification](https://huggingface.co/docs/transformers/en/tasks/sequence_classification), [averaging and ensembles](https://scikit-learn.org/stable/modules/ensemble.html), [importance-weighted cross-validation](https://jmlr.csail.mit.edu/papers/v8/sugiyama07a.html).


In [4]:
result = subprocess.run(['graphify','update','.'],cwd=ROOT,capture_output=True,text=True)
print(result.stdout)
assert result.returncode==0,result.stderr
print('Diagnostic complete. Best reported Kaggle score remains 0.98629; next-model gains are unmeasured.')


Re-extracting code files in . (no LLM needed)...
[graphify watch] No code-graph topology changes detected; outputs left untouched.
Code graph updated. For doc/paper/image changes run /graphify --update in your AI assistant.

Diagnostic complete. Best reported Kaggle score remains 0.98629; next-model gains are unmeasured.
